# 09 — Full evaluation of the metadata extraction

Run from the `notebooks/` folder after notebooks 07 and 08. It compares **four methods** field by field against your hand-checked gold file:

| method | source |
|---|---|
| `embedded` | title / authors / keywords stored in the PDF (notebook 01) |
| `llm` | the model output of notebook 07 (`eurollm_extractions.csv`) |
| `rules (08)` | DOI and document type from notebook 08, language from fastText (notebook 05, optional) |
| `combined` | `llm` for title, authors, keywords and year; DOI, type and language from the rules |

**What makes this evaluation precise**
- Every number comes with its **n** and a **95% confidence interval** (Wilson interval for proportions, bootstrap over documents for F1). With 25–40 labelled documents the intervals are wide; that is the honest uncertainty, not a flaw in the code.
- **DOI** is scored as detection *and* identification: correct, wrong DOI, spurious DOI (document has none), missed, correctly empty. Precision and recall are reported separately.
- Every DOI error is **diagnosed**: was the true DOI never found, found but outscored, or not in the extracted text at all?
- Results are broken down by **document type** and **language**, because journal articles, conference papers and reports are structured differently.
- The gold file is **checked first**: invalid values or unverified pre-filled candidates would silently corrupt every score.

Scoring rules: title correct at ≥ 90% character similarity; authors and keywords by item (authors matched on surname plus first initial, so `Smith, J.` equals `John Smith`); DOI exact after normalisation; year exact (and within ±1); type and language exact. An **empty gold cell is skipped**; `NONE` means "checked, not present".

## 1. Settings and data

In [ ]:
from pathlib import Path
import re, math, unicodedata, difflib
import numpy as np
import pandas as pd
from IPython.display import display

INVENTORY_DIR = Path('../output/inventory').resolve()
PREPROC_DIR = Path('../output/preprocessing').resolve()
OUT_DIR = Path('../output/extraction').resolve()

EVAL_SPLIT = 'dev'      # 'dev', 'test' or 'all'. Use 'test' once, at the very end.
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']
LANGS = ['en', 'nl', 'mixed']

read_csv = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents = read_csv(INVENTORY_DIR / 'documents.csv')
observations = read_csv(INVENTORY_DIR / 'metadata_observations.csv')
gold_all = read_csv(OUT_DIR / 'gold_metadata.csv')
features = read_csv(OUT_DIR / 'doc_features.csv')                  # notebook 08
cands = read_csv(OUT_DIR / 'doi_candidates.csv')                   # notebook 08
cands['page'] = pd.to_numeric(cands['page'])
llm_table = read_csv(OUT_DIR / 'eurollm_extractions.csv')          # notebook 07

gold = gold_all if EVAL_SPLIT == 'all' else gold_all[gold_all.split == EVAL_SPLIT]
ids = list(gold.document_id)
file_names = documents.set_index('document_id').file_name
print(f'Evaluating the {EVAL_SPLIT} split: {len(ids)} gold documents')

## 2. Is the gold file usable?

This table counts the filled cells and lists values that cannot be right. Fix these in `gold_metadata.csv` before trusting any score. A cell containing ` | ` is a pre-filled *candidate list* from notebook 01 that was never verified.

In [ ]:
VALID = {
    'publication_year': lambda v: bool(re.fullmatch(r'(19|20)\d{2}', v)),
    'document_type': lambda v: v in DOC_TYPES,
    'language': lambda v: v in LANGS,
    'doi': lambda v: bool(re.fullmatch(r'10\.\d{4,9}/\S+', re.sub(r'(?i)^https?://(dx\.)?doi\.org/', '', v))),
}
filled = {f: int((gold[f].str.strip() != '').sum()) for f in FIELDS}
absent = {f: int((gold[f].str.strip().str.upper() == 'NONE').sum()) for f in FIELDS}
health = pd.DataFrame({'filled': filled, 'marked NONE': absent})
health['skipped (empty)'] = len(gold) - health['filled']
display(health)

problems = []
for r in gold.itertuples(index=False):
    for f in FIELDS:
        v = getattr(r, f).strip()
        if not v or v.upper() == 'NONE':
            continue
        if ' | ' in v:
            problems.append((r.file_name, f, v[:70], 'unverified candidate list (contains " | ")'))
        elif f in VALID and not VALID[f](v):
            problems.append((r.file_name, f, v[:70], 'invalid value'))
dupes = gold[(gold.doi.str.strip() != '') & (gold.doi.str.upper() != 'NONE')]
dupes = dupes[dupes.duplicated('doi', keep=False)]
for r in dupes.itertuples(index=False):
    problems.append((r.file_name, 'doi', r.doi, 'same DOI on several documents'))
unknown = set(gold.document_id) - set(documents.document_id)
for d in unknown:
    problems.append((d, 'document_id', d, 'not in documents.csv'))

if problems:
    print(f'{len(problems)} problems in the gold file:')
    display(pd.DataFrame(problems, columns=['file', 'field', 'value', 'problem']))
else:
    print('No problems found in the gold file.')

## 3. Predictions of every method

In [ ]:
def blank():
    return {f: '' for f in FIELDS}

def from_table(df):
    table = df.set_index('document_id')
    return {d: {f: (table.at[d, f] if d in table.index and f in table.columns else '') for f in FIELDS} for d in ids}

# embedded metadata (notebook 01)
embedded = {}
for d in ids:
    row = blank()
    for field, name in [('title', 'title'), ('authors', 'author'), ('keywords', 'pdf_keywords')]:
        sub = observations[(observations.document_id == d) & (observations.field_name == name) & (observations.source_type == 'pdf_embedded')]
        row[field] = sub.observed_value.iloc[0] if len(sub) else ''
    embedded[d] = row

# LLM (notebook 07)
llm = from_table(llm_table)

# rules (notebook 08) + fastText language (notebook 05, optional)
lang_path = PREPROC_DIR / 'document_languages.csv'
fasttext = {}
if lang_path.exists():
    lang_df = read_csv(lang_path)
    fasttext = {d: l for d, l in zip(lang_df.document_id, lang_df.detected_language) if l in ('en', 'nl')}
print('fastText language available for', len(fasttext), 'documents' if fasttext else 'documents (add the export cell to notebook 05 to use it)')
feat = features.set_index('document_id')
rules = {}
for d in ids:
    row = blank()
    if d in feat.index:
        row['doi'] = feat.at[d, 'doi']
        row['document_type'] = feat.at[d, 'document_type_rule']
    row['language'] = fasttext.get(d, '')
    rules[d] = row

# combined
combined = {}
for d in ids:
    row = dict(llm[d])
    row['doi'] = rules[d]['doi']
    if rules[d]['document_type'] not in ('', 'other'):
        row['document_type'] = rules[d]['document_type']
    elif row['document_type'] not in DOC_TYPES:
        row['document_type'] = rules[d]['document_type']
    row['language'] = rules[d]['language'] or llm[d]['language']
    combined[d] = row

methods = {'embedded': embedded, 'llm': llm, 'rules (08)': rules, 'combined': combined}
print({m: sum(any(r[f] for f in FIELDS) for r in preds.values()) for m, preds in methods.items()}, '<- documents with any prediction')

## 4. Scoring one value

All metric rules live in `score_one`. It returns an `outcome` (a readable label), a `score` between 0 and 1 for aggregation, and item counts `tp`, `fp`, `fn` for the list fields.

In [ ]:
def squash(text):
    text = unicodedata.normalize('NFKD', str(text)).casefold()
    return ''.join(c for c in text if c.isalnum() and not unicodedata.combining(c))

def norm_doi(v):
    return re.sub(r'(?i)^(https?://(dx\.)?doi\.org/|doi:\s*)', '', str(v).strip()).lower().rstrip('.,;')

def split_items(value, field):
    value = str(value).strip()
    if not value or value.upper() == 'NONE':
        return []
    parts = re.split(r'\s*;\s*|\s*\|\s*', value)
    if len(parts) == 1 and field == 'authors':
        parts = re.split(r'\s+and\s+|\s*&\s*', value)
        if len(parts) == 1 and value.count(',') >= 2:
            parts = value.split(',')
    if len(parts) == 1 and field == 'keywords' and ',' in value:
        parts = value.split(',')
    return [p.strip() for p in parts if p.strip()]

def name_key(name):
    """(surname, first initial). Handles 'Last, First' and 'First Last'."""
    if ',' in name:
        last, first = [p.strip() for p in name.split(',', 1)]
    else:
        parts = name.split()
        last, first = (parts[-1] if parts else ''), ' '.join(parts[:-1])
    surname = squash(last.split()[-1]) if last.split() else ''
    return surname, squash(first)[:1]

def same_author(a, b):
    (sa, ia), (sb, ib) = name_key(a), name_key(b)
    return bool(sa) and sa == sb and (ia == ib or '' in (ia, ib))

def same_keyword(a, b):
    return squash(a) == squash(b)

def count_matches(gold_items, pred_items, same):
    used, tp = set(), 0
    for g in gold_items:
        for j, p in enumerate(pred_items):
            if j not in used and same(g, p):
                used.add(j); tp += 1
                break
    return tp

def score_one(field, g, p):
    g = '' if g.strip().upper() == 'NONE' else g.strip()
    p = str(p).strip()
    out = dict(outcome='', score=0.0, tp=0, fp=0, fn=0, sim=np.nan)
    if field == 'doi':
        gd, pdoi = norm_doi(g), norm_doi(p)
        if not gd and not pdoi: out['outcome'] = 'TN'
        elif not gd:            out['outcome'] = 'FP_spurious'
        elif not pdoi:          out['outcome'] = 'FN'
        elif gd == pdoi:        out['outcome'] = 'TP'
        else:                   out['outcome'] = 'FP_wrong'
        o = out['outcome']
        out.update(score=float(o in ('TP', 'TN')), tp=int(o == 'TP'), fp=int(o.startswith('FP')), fn=int(o in ('FN', 'FP_wrong')))
    elif field == 'title':
        sim = difflib.SequenceMatcher(None, squash(g), squash(p)).ratio() if (g or p) else 1.0
        out['sim'] = sim
        out['outcome'] = 'missing' if (g and not p) else 'correct' if sim >= 0.9 else 'close' if sim >= 0.8 else 'wrong'
        out['score'] = float(sim >= 0.9)
    elif field in ('authors', 'keywords'):
        gi, pi = split_items(g, field), split_items(p, field)
        tp = count_matches(gi, pi, same_author if field == 'authors' else same_keyword)
        out.update(tp=tp, fp=len(pi) - tp, fn=len(gi) - tp)
        denom = 2 * tp + out['fp'] + out['fn']
        out['score'] = 1.0 if denom == 0 else 2 * tp / denom
        out['outcome'] = 'missing' if (gi and not pi) else 'correct' if (out['fp'] == 0 and out['fn'] == 0) else 'partial'
    elif field == 'publication_year':
        if g and not p: out['outcome'] = 'missing'
        elif g == p:    out['outcome'] = 'correct'
        elif g.isdigit() and p.isdigit() and abs(int(g) - int(p)) <= 1: out['outcome'] = 'off_by_one'
        else:           out['outcome'] = 'wrong'
        out['score'] = float(out['outcome'] == 'correct')
        out['sim'] = float(out['outcome'] in ('correct', 'off_by_one'))     # reused as "within +-1"
    else:                                                                     # document_type, language
        allowed = set(DOC_TYPES) if field == 'document_type' else set(LANGS)
        if g and not p:             out['outcome'] = 'missing'
        elif p and p not in allowed: out['outcome'] = 'invalid_label'
        elif squash(g) == squash(p): out['outcome'] = 'correct'
        else:                        out['outcome'] = 'wrong'
        out['score'] = float(out['outcome'] == 'correct')
    return out

# self-tests
assert score_one('authors', 'Cosoveanu, Flavia Simona; Buijs, Jean-Marie', 'Flavia Cosoveanu; J.-M. Buijs')['score'] == 1.0
assert score_one('doi', '10.1/ABC', 'https://doi.org/10.1/abc.')['outcome'] == 'TP'
assert score_one('doi', 'NONE', '10.1/abc')['outcome'] == 'FP_spurious'
assert score_one('doi', '10.1/abc', '10.1/xyz')['outcome'] == 'FP_wrong'
assert score_one('title', 'Adaptive Capacities for Diversified Flood Risk', 'Adaptive capacities for diversiﬁed flood risk')['outcome'] == 'correct'
assert score_one('document_type', 'journal_article', 'whitepaper')['outcome'] == 'invalid_label'
print('Scoring self-tests passed')

## 5. Score every method on every labelled field

In [ ]:
records = []
gold_rows = gold.set_index('document_id').to_dict('index')
for method, preds in methods.items():
    for d in ids:
        for field in FIELDS:
            g = gold_rows[d][field]
            if g.strip() == '':
                continue                                           # not labelled: skipped
            records.append({'method': method, 'document_id': d, 'file_name': file_names.get(d, d), 'field': field,
                            'gold': g.strip(), 'pred': str(preds[d][field]).strip(), **score_one(field, g, preds[d][field])})
records = pd.DataFrame(records)
print(f'{len(records)} scored values;', records.groupby('field').document_id.nunique().to_dict())

## 6. Metrics with confidence intervals

`value` is the metric, `ci_low`/`ci_high` its 95% interval, `n` the number of labelled documents behind it. Precision and recall for DOI use their own denominators (predicted DOIs and true DOIs).

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else np.nan
    r = tp / (tp + fn) if tp + fn else np.nan
    f = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    return p, r, f

def bootstrap_f1(tp, fp, fn, n_boot=2000, seed=0):
    tp, fp, fn = map(np.asarray, (tp, fp, fn))
    n = len(tp)
    if n == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        vals.append(prf(tp[i].sum(), fp[i].sum(), fn[i].sum())[2])
    vals = [v for v in vals if not np.isnan(v)]
    return tuple(np.percentile(vals, [2.5, 97.5])) if vals else (np.nan, np.nan)

def summarize(records):
    rows = []
    def add(method, field, metric, value, lo, hi, n):
        rows.append(dict(method=method, field=field, metric=metric, value=value, ci_low=lo, ci_high=hi, n=n))
    for (method, field), g in records.groupby(['method', 'field']):
        n = len(g)
        if field == 'title':
            for thr in (0.9, 0.8):
                k = int((g.sim >= thr).sum()); lo, hi = wilson(k, n)
                add(method, field, f'accuracy@{thr:.2f}', k / n, lo, hi, n)
            add(method, field, 'mean similarity', g.sim.mean(), np.nan, np.nan, n)
            add(method, field, 'missing rate', (g.outcome == 'missing').mean(), *wilson(int((g.outcome == 'missing').sum()), n), n)
        elif field in ('authors', 'keywords'):
            p, r, f = prf(g.tp.sum(), g.fp.sum(), g.fn.sum()); lo, hi = bootstrap_f1(g.tp, g.fp, g.fn)
            add(method, field, 'precision', p, np.nan, np.nan, n)
            add(method, field, 'recall', r, np.nan, np.nan, n)
            add(method, field, 'F1', f, lo, hi, n)
            k = int((g.outcome == 'correct').sum())
            add(method, field, 'exact list match', k / n, *wilson(k, n), n)
            add(method, field, 'missing rate', (g.outcome == 'missing').mean(), *wilson(int((g.outcome == 'missing').sum()), n), n)
        elif field == 'doi':
            c = g.outcome.value_counts()
            TP, FPw, FPs, FN, TN = (int(c.get(k, 0)) for k in ('TP', 'FP_wrong', 'FP_spurious', 'FN', 'TN'))
            p, r, f = prf(TP, FPw + FPs, FN + FPw)
            add(method, field, 'precision', p, *wilson(TP, TP + FPw + FPs), TP + FPw + FPs)
            add(method, field, 'recall', r, *wilson(TP, TP + FN + FPw), TP + FN + FPw)
            add(method, field, 'F1', f, *bootstrap_f1(g.tp, g.fp, g.fn), n)
            add(method, field, 'accuracy', (TP + TN) / n, *wilson(TP + TN, n), n)
            for name, v in (('TP', TP), ('wrong DOI', FPw), ('spurious DOI', FPs), ('missed', FN), ('correctly empty', TN)):
                add(method, field, f'count: {name}', v, np.nan, np.nan, n)
        elif field == 'publication_year':
            k = int((g.outcome == 'correct').sum()); add(method, field, 'accuracy', k / n, *wilson(k, n), n)
            k1 = int((g.sim == 1).sum());            add(method, field, 'within +-1 year', k1 / n, *wilson(k1, n), n)
            add(method, field, 'missing rate', (g.outcome == 'missing').mean(), *wilson(int((g.outcome == 'missing').sum()), n), n)
        else:
            k = int((g.outcome == 'correct').sum()); add(method, field, 'accuracy', k / n, *wilson(k, n), n)
            add(method, field, 'missing rate', (g.outcome == 'missing').mean(), *wilson(int((g.outcome == 'missing').sum()), n), n)
            add(method, field, 'invalid labels', int((g.outcome == 'invalid_label').sum()), np.nan, np.nan, n)
    return pd.DataFrame(rows)

summary = summarize(records)
HEADLINE = {'title': 'accuracy@0.90', 'authors': 'F1', 'keywords': 'F1', 'doi': 'F1',
            'publication_year': 'accuracy', 'document_type': 'accuracy', 'language': 'accuracy'}

def fmt(row):
    if row is None or pd.isna(row.value):
        return '-'
    if row.metric.startswith('count') or row.metric == 'invalid labels':
        return str(int(row.value))
    text = f'{row.value:.2f}'
    return text + (f' [{row.ci_low:.2f}-{row.ci_high:.2f}]' if not pd.isna(row.ci_low) else '')

head = []
for field, metric in HEADLINE.items():
    sub = summary[(summary.field == field) & (summary.metric == metric)]
    line = {'field': field, 'metric': metric, 'n': int(sub.n.max()) if len(sub) else 0}
    for method in methods:
        m = sub[sub.method == method]
        line[method] = fmt(m.iloc[0]) if len(m) else '-'
    head.append(line)
print(f'Headline results, {EVAL_SPLIT} split (value [95% interval]):')
display(pd.DataFrame(head).set_index('field'))

In [ ]:
# All metrics, one table per field
for field in HEADLINE:
    sub = summary[summary.field == field]
    if sub.empty:
        continue
    table = sub.assign(cell=sub.apply(fmt, axis=1)).pivot(index='metric', columns='method', values='cell')
    print(f'\n{field.upper()}  (n = {int(sub.n.max())})')
    display(table[[m for m in methods if m in table.columns]])

## 7. DOI in detail

The questions that matter: **where** is the true DOI printed, and **why** does an extraction fail?

*7.1* shows the location of the true DOI for each document type (from the candidates harvested in notebook 08). *7.2* classifies every DOI error of the `combined` method.

In [ ]:
LOCATION_ORDER = ['METADATA', 'LINK', 'HEADER', 'FOOTER', 'SIDEBAR_LEFT', 'SIDEBAR_RIGHT',
                  'MARGIN_LEFT', 'MARGIN_RIGHT', 'BODY', 'REFERENCES']
type_of = {}
for d in ids:
    g_type = gold_rows[d]['document_type'].strip()
    type_of[d] = g_type if g_type in DOC_TYPES else (feat.at[d, 'document_type_rule'] + ' (rule)' if d in feat.index else 'unknown')

# page text of notebook 05, to tell "not harvested" from "not in the text at all"
pages_path = PREPROC_DIR / 'pages.jsonl'
page_text = {}
if pages_path.exists():
    pages = pd.read_json(pages_path, lines=True)
    pages = pages[pages.document_id.isin(ids)]
    page_text = {d: squash(' '.join(g.sort_values('page_number').original_text)) for d, g in pages.groupby('document_id')}

def true_doi_location(d, true_doi):
    sightings = cands[(cands.document_id == d) & (cands.doi == true_doi)]
    if len(sightings):
        best = min(sightings.itertuples(), key=lambda c: (LOCATION_ORDER.index(c.zone), c.page))
        return best.zone, int(best.page)
    if d in page_text and squash(true_doi) in page_text[d]:
        return 'in text, not harvested', None
    return 'not in extracted text', None

rows = []
for d in ids:
    g = gold_rows[d]['doi'].strip()
    if g and g.upper() != 'NONE':
        zone, page = true_doi_location(d, norm_doi(g))
        rows.append({'document_id': d, 'file_name': file_names[d], 'type': type_of[d], 'location': zone, 'page': page})
loc = pd.DataFrame(rows)
if loc.empty:
    print('No gold DOIs to analyse.')
else:
    print(f'7.1 Where the true DOI is printed ({len(loc)} documents that have a DOI):')
    order = [z for z in LOCATION_ORDER + ['in text, not harvested', 'not in extracted text'] if z in set(loc.location)]
    display(pd.crosstab(loc.type, loc.location).reindex(columns=order, fill_value=0))
    print('Page of the first sighting:')
    display(loc.dropna(subset=['page']).groupby(['type', 'page']).size().unstack(fill_value=0))

In [ ]:
def diagnose(d, g, p):
    """Returns (category, detail) for one DOI error."""
    gd, pdoi = ('' if g.upper() == 'NONE' else norm_doi(g)), norm_doi(p)
    if not gd and pdoi:
        f = feat.loc[d] if d in feat.index else None
        detail = f'zone {f.doi_zone}, page {f.doi_page}, evidence: {f.doi_evidence}' if f is not None and f.doi_zone else ''
        return 'spurious DOI (document has none)', detail
    zone, page = true_doi_location(d, gd)
    where = f'{zone}, page {page}' if page else zone
    if gd and not pdoi:
        if zone == 'in text, not harvested': return 'missed: in the text, but not harvested', 'check the DOI format / line breaks'
        if zone == 'not in extracted text':  return 'missed: not in the extracted text', 'image, OCR, or an error in the gold file'
        return 'missed: harvested but below MIN_SCORE', f'true DOI seen at {where}'
    if zone in ('in text, not harvested', 'not in extracted text'):
        return 'wrong DOI: true DOI never harvested', f'true DOI: {zone}'
    return 'wrong DOI: true DOI was outscored', f'true DOI seen at {where}'

doi_errors = records[(records.field == 'doi') & (records.method == 'combined') & (records.score < 1)].copy()
diag = [diagnose(r.document_id, r.gold, r.pred) for r in doi_errors.itertuples()]
doi_errors['category'] = [c for c, _ in diag]
doi_errors['detail'] = [t for _, t in diag]
doi_errors['type'] = doi_errors.document_id.map(type_of)
print(f'7.2 DOI errors of the combined method: {len(doi_errors)}')
if len(doi_errors):
    display(doi_errors.category.value_counts().to_frame('documents'))
    display(doi_errors[['file_name', 'type', 'gold', 'pred', 'category', 'detail']])

print('\n7.3 DOI outcome by document type (rows = type):')
for method in ('llm', 'combined'):
    sub = records[(records.field == 'doi') & (records.method == method)]
    print(f'\n{method}')
    display(pd.crosstab(sub.document_id.map(type_of).rename('type'), sub.outcome))

## 8. Breakdown by document type and by language

Mean score per group (`n` = documents in the group). Title, DOI, year, type and language are 0/1 per document; authors and keywords use the per-document F1. Groups with very few documents say little: look at `n`.

In [ ]:
lang_of = {d: gold_rows[d]['language'].strip() or fasttext.get(d, '') or 'unknown' for d in ids}
rec = records.assign(type=records.document_id.map(type_of), lang=records.document_id.map(lang_of))

def grouped(rec, by):
    g = rec.groupby(['field', by, 'method']).score.agg(['mean', 'count']).reset_index()
    g['cell'] = g.apply(lambda r: f'{r["mean"]:.2f} (n={int(r["count"])})', axis=1)
    return g.pivot(index=['field', by], columns='method', values='cell').fillna('-')[[m for m in methods]]

print('By document type (gold type where labelled, otherwise the notebook 08 guess):')
display(grouped(rec, 'type'))
print('By language:')
display(grouped(rec, 'lang'))

## 9. Document type and language: confusion matrices

In [ ]:
for field in ('document_type', 'language'):
    for method in methods:
        sub = records[(records.field == field) & (records.method == method)]
        if sub.empty or (sub.pred == '').all():
            continue
        print(f'\n{field} - {method}  (rows = gold, columns = predicted)')
        cm = pd.crosstab(sub.gold, sub.pred.replace('', '(none)'))
        display(cm)
        stats = pd.DataFrame({'precision': {c: cm.at[c, c] / cm[c].sum() if c in cm.index and c in cm.columns and cm[c].sum() else np.nan for c in cm.columns},
                              'recall': {c: cm.at[c, c] / cm.loc[c].sum() if c in cm.index and c in cm.columns else np.nan for c in cm.index}})
        display(stats.round(2))

## 10. Error list and saved results

Every scored value that is not fully correct, for the method chosen in `ERROR_METHOD`. Work through it: each row is either a prompt/rule problem or a mistake in the gold file.

In [ ]:
ERROR_METHOD = 'combined'
errors = records[(records.method == ERROR_METHOD) & (records.score < 1)].copy()
errors['type'] = errors.document_id.map(type_of)
errors['gold'] = errors.gold.str.slice(0, 80)
errors['pred'] = errors.pred.str.slice(0, 80)
print(f'{len(errors)} values not fully correct for "{ERROR_METHOD}":')
display(errors[['file_name', 'type', 'field', 'gold', 'pred', 'outcome']].sort_values(['field', 'file_name']).reset_index(drop=True))

records.to_csv(OUT_DIR / f'evaluation_records_{EVAL_SPLIT}.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUT_DIR / f'evaluation_summary_{EVAL_SPLIT}.csv', index=False, encoding='utf-8-sig')
errors.to_csv(OUT_DIR / f'evaluation_errors_{EVAL_SPLIT}.csv', index=False, encoding='utf-8-sig')
print('Saved evaluation_records / evaluation_summary / evaluation_errors for the', EVAL_SPLIT, 'split')